Import libraries

In [16]:
import requests
import pandas as pd
import sqlite3

Fetch data from API

In [17]:
users = requests.get("https://jsonplaceholder.typicode.com/users").json()
posts = requests.get("https://jsonplaceholder.typicode.com/posts").json()

print("Users:", len(users))
print("Posts:", len(posts))

Users: 10
Posts: 100


Convert to DataFrames

In [20]:
df_users = pd.DataFrame(users)
df_posts = pd.DataFrame(posts)

df_users.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


Basic data cleaning

In [21]:
df_users['city'] = df_users['address'].apply(lambda x: x['city'])

In [22]:
df_users = df_users[['id', 'name', 'email', 'city']]

In [23]:
df_posts = df_posts[['userId', 'id', 'title']]

In [24]:
df_users.head()

,id,name,email,city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview


Create SQL database

In [25]:
conn = sqlite3.connect("user_data.db")

Load data into SQL

In [26]:
df_users.to_sql("users", conn, if_exists="replace", index=False)
df_posts.to_sql("posts", conn, if_exists="replace", index=False)

100

Run SQL queries

In [27]:
#Top users by posts
query = """
SELECT userId, COUNT(*) as total_posts
FROM posts
GROUP BY userId
ORDER BY total_posts DESC
"""

result = pd.read_sql(query, conn)
result

,userId,total_posts
0,10,10
1,9,10
2,8,10
3,7,10
4,6,10
5,5,10
6,4,10
7,3,10
8,2,10
9,1,10


In [28]:
#Join users + posts
query = """
SELECT u.name, COUNT(p.id) as total_posts
FROM users u
JOIN posts p ON u.id = p.userId
GROUP BY u.name
ORDER BY total_posts DESC
"""

pd.read_sql(query, conn)

,name,total_posts
0,Patricia Lebsack,10
1,Nicholas Runolfsdottir V,10
2,Mrs. Dennis Schulist,10
3,Leanne Graham,10
4,Kurtis Weissnat,10
5,Glenna Reichert,10
6,Ervin Howell,10
7,Clementine Bauch,10
8,Clementina DuBuque,10
9,Chelsey Dietrich,10


In [29]:
#Users per city
query = """
SELECT city, COUNT(*) as total_users
FROM users
GROUP BY city
"""

pd.read_sql(query, conn)

,city,total_users
0,Aliyaview,1
1,Bartholomebury,1
2,Gwenborough,1
3,Howemouth,1
4,Lebsackbury,1
5,McKenziehaven,1
6,Roscoeview,1
7,South Christy,1
8,South Elvis,1
9,Wisokyburgh,1
